# S13 probe — how long will the real run take?\n\nThree training runs of 50M tokens are about to be booked on this GPU. This notebook measures throughput, peak memory and maximum batch size first, so the real harness can be sized instead of guessed at.\n\n**Run all, then paste the final JSON back.** Two or three minutes.

In [10]:
# The reversible engine, embedded so this notebook needs nothing else. Written to disk
# rather than pasted into a cell so the import below is the same module the repo tests.
engine_src = r'''"""The reversible stack: the thing §16 of the session describes, built so it can be checked.

A standard residual block adds its output to its own input, `p = p + f(p)`, and that cannot be
run backwards: recovering the input would need `f` evaluated *at the input being recovered*.
The rules here reach further back so that the block is always evaluated at a state the
backward pass already holds, which makes the stack invertible and lets the forward pass throw
away every intermediate activation.

Nothing here is approximate. The whole point of the exercise is that a reversible stack which
is subtly wrong still trains — the loss still falls — so `gradient_check()` below compares
against ordinary autograd rather than trusting the loss curve.

Imported by `s13_reversibility.py`; kept separate so it can be tested without running the
harness, and so the Colab notebook has one file to carry.
"""

import math
from contextlib import contextmanager

import torch
import torch.nn as nn
import torch.nn.functional as F

RULES = ("standard", "euler", "midpoint", "blended", "revnet")
REVERSIBLE = ("midpoint", "blended", "revnet")       # euler and standard are not invertible


# --------------------------------------------------------------------------------------
# Model. The S9-S12 decoder, with one change forced by this session: no dropout anywhere.
# §16 makes that a correctness requirement rather than a tuning choice — the backward pass
# recomputes each block, and a random mask would make the reconstruction differ from what
# the forward pass actually used.
# --------------------------------------------------------------------------------------
class Config:
    def __init__(self, **kw):
        self.vocab_size, self.d_model, self.n_layer, self.d_head = 256, 384, 10, 64
        self.block_size = 512
        self.embed_std = 0.02
        self.rule, self.h, self.gamma = "standard", 1.0, 0.0
        self.__dict__.update(kw)
        self.n_head = max(1, self.d_model // self.d_head)
        self.d_ff = getattr(self, "d_ff", None) or round(self.d_model * 8 / 3 / 64) * 64

    def __repr__(self):
        return (f"Config(rule={self.rule}, d_model={self.d_model}, n_layer={self.n_layer}, "
                f"d_ff={self.d_ff}, vocab={self.vocab_size}, h={self.h}, gamma={self.gamma})")


class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))

    def forward(self, x):
        # Accumulate in float32 for the low-precision dtypes, but never DOWN-cast: writing
        # `x.float()` unconditionally computes the norm in float32 even for a float64 input,
        # which quantises the block into steps of ~1e-7. Any reconstruction error smaller
        # than a step can then flip one, so the block behaves discontinuously and the
        # reversible path looks broken when it is not.
        acc = x.float() if x.dtype in (torch.bfloat16, torch.float16) else x
        return x * torch.rsqrt(acc.pow(2).mean(-1, keepdim=True) + 1e-6).to(x.dtype) * self.g


class Block(nn.Module):
    """Attention followed by the feed-forward network — §16's `f`, the whole transformer block.

    `width` is the channel count this block reads and writes. It equals d_model for every rule
    except `revnet`, where the stream is split in half and each coupling function sees d_model/2.
    """

    def __init__(self, cfg, width=None):
        super().__init__()
        d = width or cfg.d_model
        self.n_head = max(1, d // cfg.d_head)
        self.d_head = cfg.d_head
        H = self.n_head * self.d_head
        d_ff = round(d * 8 / 3 / 64) * 64 if width else cfg.d_ff
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.qkv = nn.Linear(d, 3 * H, bias=False)
        self.proj = nn.Linear(H, d, bias=False)
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        B, T, _ = x.shape
        H = self.n_head * self.d_head
        q, k, v = self.qkv(self.n1(x)).split(H, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = self.proj(a.transpose(1, 2).reshape(B, T, H))
        y = self.n2(x)
        return x + self.down(F.silu(self.gate(y)) * self.up(y))


# --------------------------------------------------------------------------------------
# The five update rules, each written once so the forward pass and the inverse cannot drift
# apart. `step` advances the stack; `invert` recovers the state two layers back.
# --------------------------------------------------------------------------------------
def step(rule, p_prev, p, fout, h, gamma):
    """One layer of the stack. `fout` is f(p), already computed."""
    if rule == "standard":
        return p + fout                       # the ordinary residual; p_prev unused
    if rule == "euler":
        return p + h * fout                   # a step size, still not invertible
    if rule == "midpoint":
        return p_prev + 2 * h * fout          # §16's rule
    if rule == "blended":
        return (1 - gamma) * p_prev + gamma * p + 2 * h * fout
    raise ValueError(rule)


def invert(rule, p_next, p, fout, h, gamma):
    """Recover p_prev from p_next and p. `fout` is f(p) — a state the backward pass holds."""
    if rule == "midpoint":
        return p_next - 2 * h * fout
    if rule == "blended":
        # The division is the whole problem with this rule: it multiplies any error in
        # p_next by 1/(1-gamma), once per layer. See the gamma sweep in the harness.
        return (p_next - gamma * p - 2 * h * fout) / (1 - gamma)
    raise ValueError(f"{rule} has no inverse")


def _grad_recursion(rule, A, B, jvp, h, gamma):
    """Push the gradient pair (dL/dp_next, dL/dp) down one layer.

    p_next = c1*p_prev + c2*p + k*f(p), so dL/dp_prev = c1*A and dL/dp = B + c2*A + k*J^T A.
    `jvp` is the J^T A term, already scaled by k.
    """
    if rule == "midpoint":
        return B + jvp, A                     # c1 = 1, c2 = 0
    if rule == "blended":
        return B + gamma * A + jvp, (1 - gamma) * A
    raise ValueError(rule)


# --------------------------------------------------------------------------------------
# Stored-activation path: ordinary autograd, every intermediate kept. The reference.
# --------------------------------------------------------------------------------------
def run_stored(blocks, x, rule, h, gamma):
    if rule == "revnet":
        x1, x2 = x.chunk(2, dim=-1)
        for l in range(0, len(blocks) - 1, 2):
            y1 = x1 + blocks[l](x2)
            y2 = x2 + blocks[l + 1](y1)
            x1, x2 = y1, y2
        return torch.cat([x1, x2], dim=-1)
    p_prev = x
    p = p_prev + h * blocks[0](p_prev)         # bootstrap: the first step has no p_prev to use
    for l in range(1, len(blocks)):
        p_prev, p = p, step(rule, p_prev, p, blocks[l](p), h, gamma)
    return p


# --------------------------------------------------------------------------------------
# Reversible path: the forward keeps only the boundary states.
# --------------------------------------------------------------------------------------
class _RevStack(torch.autograd.Function):
    """Forward under no_grad, keeping x, p_L and p_{L-1}. Backward rebuilds the rest.

    The input state x is kept because the bootstrap step is a plain Euler step and is not
    itself invertible — which is exactly §16's "the state that enters the stack".
    """

    @staticmethod
    def forward(ctx, x, blocks, rule, h, gamma, *params):
        ctx.blocks, ctx.rule, ctx.h, ctx.gamma = blocks, rule, h, gamma
        with torch.no_grad():
            p_prev = x
            p = p_prev + h * blocks[0](p_prev)
            for l in range(1, len(blocks)):
                p_prev, p = p, step(rule, p_prev, p, blocks[l](p), h, gamma)
        ctx.save_for_backward(x, p, p_prev)
        return p

    @staticmethod
    def backward(ctx, g_out):
        x, p, p_prev = ctx.saved_tensors
        blocks, rule, h, gamma = ctx.blocks, ctx.rule, ctx.h, ctx.gamma
        p_hi, p_lo = p, p_prev                        # (p_{l+1}, p_l)
        A, B = g_out, torch.zeros_like(g_out)         # dL/dp_{l+1}, dL/dp_l
        pgrads = [torch.zeros_like(q) for b in blocks for q in b.parameters()]
        sizes = [len(list(b.parameters())) for b in blocks]
        offs, acc = [], 0
        for n in sizes:
            offs.append(acc)
            acc += n

        for l in range(len(blocks) - 1, 0, -1):
            plist = list(blocks[l].parameters())
            pl = p_lo.detach().requires_grad_(True)
            with torch.enable_grad():
                fout = blocks[l](pl)
            with torch.no_grad():
                p_down = invert(rule, p_hi, p_lo, fout.detach(), h, gamma)
            g = torch.autograd.grad(fout, [pl] + plist, grad_outputs=2 * h * A)
            for i, gv in enumerate(g[1:]):
                pgrads[offs[l] + i] += gv
            A, B = _grad_recursion(rule, A, B, g[0], h, gamma)
            p_hi, p_lo = p_lo, p_down

        # the bootstrap layer, whose input is the x we kept
        x0 = x.detach().requires_grad_(True)
        plist = list(blocks[0].parameters())
        with torch.enable_grad():
            f0 = blocks[0](x0)
        g = torch.autograd.grad(f0, [x0] + plist, grad_outputs=h * A)
        for i, gv in enumerate(g[1:]):
            pgrads[offs[0] + i] += gv
        # p_1 = p_0 + h*f_0(p_0), so p_0 receives A through the identity path and h*J^T A
        # through the block — on top of B, which layer 1 already accumulated into it. The
        # B term is easy to drop, and dropping it leaves every block gradient correct while
        # silently corrupting only the embedding gradients.
        gx = B + A + g[0]
        return (gx, None, None, None, None, *pgrads)


class _RevNetStack(torch.autograd.Function):
    """Channel-coupling reversibility: y1 = x1 + F(x2), y2 = x2 + G(y1).

    Each layer inverts on its own, exactly, with no step size and no stability band — which
    is why this is the variant that still trains when the midpoint rule will not.
    """

    @staticmethod
    def forward(ctx, x, blocks, *params):
        ctx.blocks = blocks
        with torch.no_grad():
            x1, x2 = x.chunk(2, dim=-1)
            for l in range(0, len(blocks) - 1, 2):
                x1 = x1 + blocks[l](x2)
                x2 = x2 + blocks[l + 1](x1)
        out = torch.cat([x1, x2], dim=-1)
        ctx.save_for_backward(out)
        return out

    @staticmethod
    def backward(ctx, g_out):
        (out,) = ctx.saved_tensors
        blocks = ctx.blocks
        y1, y2 = out.chunk(2, dim=-1)
        g1, g2 = g_out.chunk(2, dim=-1)
        g1, g2 = g1.contiguous(), g2.contiguous()
        pgrads = [torch.zeros_like(q) for b in blocks for q in b.parameters()]
        sizes = [len(list(b.parameters())) for b in blocks]
        offs, acc = [], 0
        for n in sizes:
            offs.append(acc)
            acc += n

        for l in range(len(blocks) - 2, -1, -2):
            # undo y2 = x2 + G(y1)
            y1d = y1.detach().requires_grad_(True)
            with torch.enable_grad():
                gy = blocks[l + 1](y1d)
            with torch.no_grad():
                x2 = y2 - gy.detach()
            gg = torch.autograd.grad(gy, [y1d] + list(blocks[l + 1].parameters()),
                                     grad_outputs=g2)
            for i, gv in enumerate(gg[1:]):
                pgrads[offs[l + 1] + i] += gv
            g1 = g1 + gg[0]
            # undo y1 = x1 + F(x2)
            x2d = x2.detach().requires_grad_(True)
            with torch.enable_grad():
                fx = blocks[l](x2d)
            with torch.no_grad():
                x1 = y1 - fx.detach()
            gf = torch.autograd.grad(fx, [x2d] + list(blocks[l].parameters()),
                                     grad_outputs=g1)
            for i, gv in enumerate(gf[1:]):
                pgrads[offs[l] + i] += gv
            g2 = g2 + gf[0]
            y1, y2 = x1, x2
        return (torch.cat([g1, g2], dim=-1), None, *pgrads)


def run_reversible(blocks, x, rule, h, gamma):
    params = [q for b in blocks for q in b.parameters()]
    if rule == "revnet":
        return _RevNetStack.apply(x, blocks, *params)
    return _RevStack.apply(x, blocks, rule, h, gamma, *params)


# --------------------------------------------------------------------------------------
class Model(nn.Module):
    def __init__(self, cfg, reversible=False):
        super().__init__()
        self.cfg, self.reversible = cfg, reversible
        width = cfg.d_model // 2 if cfg.rule == "revnet" else cfg.d_model
        n = cfg.n_layer + (cfg.n_layer % 2 if cfg.rule == "revnet" else 0)
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos = nn.Embedding(cfg.block_size, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg, width if cfg.rule == "revnet" else None)
                                    for _ in range(n))
        self.norm_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 1.0 / math.sqrt(m.weight.shape[1]))
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, cfg.embed_std)

    def forward(self, idx):
        c = self.cfg
        x = self.embed(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        runner = run_reversible if self.reversible else run_stored
        x = runner(self.blocks, x, c.rule, c.h, c.gamma)
        return self.head(self.norm_f(x))

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


# --------------------------------------------------------------------------------------
# Measurement. Two different quantities, and the difference matters.
#   * activation bytes — what the forward pass kept for the backward pass. This is the
#     quantity §1's 127.5 GiB and §17's 1.5 GiB are about, and it is device-independent.
#   * peak memory — what the assignment asks for: everything resident at the high-water
#     mark, weights and optimizer included. CUDA only.
# --------------------------------------------------------------------------------------
@contextmanager
def activation_bytes():
    """Sum the unique storages autograd saves for backward, via saved-tensor hooks."""
    seen, total = {}, [0]

    def pack(t):
        st = t.untyped_storage()
        key = st.data_ptr()
        if key and key not in seen:
            seen[key] = st.size()
            total[0] += st.size()
        return t

    def unpack(t):
        return t

    with torch.autograd.graph.saved_tensors_hooks(pack, unpack):
        yield total


def peak_bytes_reset(device):
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)


def peak_bytes_read(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        return torch.cuda.max_memory_allocated(device)
    return None


# --------------------------------------------------------------------------------------
def gradient_check(cfg, batch=2, seq=32, seed=0, device="cpu", dtype=torch.float32):
    """The gate that matters: do reversible gradients equal stored-activation gradients?

    A reversible stack that reconstructs slightly wrong still trains, and its loss still
    falls, so the loss curve is not evidence of anything. This is.
    """
    device = torch.device(device)
    torch.manual_seed(seed)
    ref = Model(cfg, reversible=False).to(device=device, dtype=dtype)
    torch.manual_seed(seed)
    rev = Model(cfg, reversible=True).to(device=device, dtype=dtype)

    g = torch.Generator(device="cpu").manual_seed(seed + 1)
    idx = torch.randint(cfg.vocab_size, (batch, seq), generator=g).to(device)
    tgt = torch.randint(cfg.vocab_size, (batch, seq), generator=g).to(device)

    def loss_of(model, counter=False):
        logits = model(idx)
        return F.cross_entropy(logits.float().reshape(-1, cfg.vocab_size), tgt.reshape(-1))

    # Count only what the FORWARD pass saved. Leaving the hook installed during backward
    # counts the one-layer graphs the reversible path rebuilds, which is the memory it
    # explicitly does not hold on to, and makes the saving look far smaller than it is.
    with activation_bytes() as a_ref:
        l_ref = loss_of(ref)
    l_ref.backward()
    with activation_bytes() as a_rev:
        l_rev = loss_of(rev)
    l_rev.backward()

    worst, worst_name = 0.0, ""
    for (n, a), (_, b) in zip(ref.named_parameters(), rev.named_parameters()):
        if a.grad is None or b.grad is None:
            continue
        scale = max(a.grad.abs().max().item(), 1e-12)
        rel = (a.grad - b.grad).abs().max().item() / scale
        if rel > worst:
            worst, worst_name = rel, n
    return {
        "rule": cfg.rule, "n_layer": cfg.n_layer,
        "worst_rel_grad_error": worst, "worst_param": worst_name,
        "activation_bytes_stored": a_ref[0], "activation_bytes_reversible": a_rev[0],
        "activation_ratio": a_ref[0] / max(a_rev[0], 1),
        "params": ref.n_params(),
    }


if __name__ == "__main__":
    # Runnable self-check. Two separate questions, and conflating them is how a correct
    # implementation gets mistaken for a broken one:
    #
    #   1. Is the implementation exact?  Ask in float64. An exact reversible stack differs
    #      from ordinary autograd only by rounding, so the error must collapse when the
    #      precision rises. If it does not, the arithmetic is wrong.
    #   2. How much does it drift in practice?  Ask in float32. This error is real and it
    #      grows with depth, because every reconstruction feeds the next one.
    import torch

    print("1. exactness  (float64 — must collapse to rounding)")
    print(f"   {'rule':<10}{'layers':>7}{'worst rel grad err':>21}{'':>4}")
    bad = []
    for rule in REVERSIBLE:
        for n_layer in (4, 12):
            cfg = Config(rule=rule, n_layer=n_layer, d_model=128, vocab_size=512,
                         block_size=64, h=0.25 if rule != "revnet" else 1.0,
                         gamma=0.5 if rule == "blended" else 0.0)
            e = gradient_check(cfg, batch=2, seq=16, dtype=torch.float64)["worst_rel_grad_error"]
            ok = e < 1e-9
            bad += [] if ok else [(rule, n_layer, e)]
            print(f"   {rule:<10}{n_layer:>7}{e:>21.2e}{'  ok' if ok else '  FAIL':>4}")

    print("\n2. drift and memory  (float32 — the practical numbers)")
    print(f"   {'rule':<10}{'layers':>7}{'drift':>12}{'act bytes stored':>19}"
          f"{'reversible':>12}{'ratio':>9}")
    for rule in REVERSIBLE:
        for n_layer in (4, 12):
            cfg = Config(rule=rule, n_layer=n_layer, d_model=128, vocab_size=512,
                         block_size=64, h=0.25 if rule != "revnet" else 1.0,
                         gamma=0.5 if rule == "blended" else 0.0)
            r = gradient_check(cfg, batch=2, seq=16)
            print(f"   {rule:<10}{n_layer:>7}{r['worst_rel_grad_error']:>12.2e}"
                  f"{r['activation_bytes_stored']:>19,}{r['activation_bytes_reversible']:>12,}"
                  f"{r['activation_ratio']:>8.1f}x")

    print("\n3. euler is not reversible, which is the point of including it")
    cfg = Config(rule="euler", n_layer=8, d_model=128, vocab_size=512, block_size=64, h=0.25)
    try:
        invert("euler", torch.zeros(1), torch.zeros(1), torch.zeros(1), 0.25, 0.0)
        print("   FAIL: euler claimed an inverse")
        bad.append(("euler", 8, float("inf")))
    except ValueError as exc:
        print(f"   {exc}")

    print("\nOK" if not bad else f"\nFAILED: {bad}")
'''
open('s13_reversible.py', 'w').write(engine_src)
print(f'wrote s13_reversible.py ({len(engine_src):,} bytes)')


wrote s13_reversible.py (19,975 bytes)


## 1 · What hardware is this, and what precision can it do?

In [11]:
import subprocess, torch, platform
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("no nvidia-smi on this machine")
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {dev}")
if dev.type == "cuda":
    cap = torch.cuda.get_device_capability()
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"compute capability {cap[0]}.{cap[1]} | VRAM {total/1e9:.1f} GB")
    # bf16 needs Ampere (8.0+). A T4 is Turing (7.5), so the low-precision work has to be
    # fp16 there, and the harness must record which one it actually used.
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
    print(f"fp16 supported: True (all CUDA)")


Tesla T4, 15360 MiB, 580.82.07
torch 2.11.0+cu128 | device cuda
compute capability 7.5 | VRAM 15.6 GB
bf16 supported: True
fp16 supported: True (all CUDA)


In [12]:
dev_name = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2 · Does the reversible stack work on this device?\n\nThe custom `autograd.Function` has only ever run on CPU. If reversible gradients do not match ordinary autograd here, nothing measured below means anything.

In [13]:
import torch, s13_reversible as R

# Does the custom autograd.Function work on CUDA at all? Everything downstream assumes it.
# float64 is the exactness check; a T4 is slow at it but this is 2x16 tokens.
print(f"{'rule':<10}{'device':>8}{'dtype':>10}{'worst rel grad err':>21}")
for rule in R.REVERSIBLE:
    for dv, dt in ((dev_name, torch.float32), ("cpu", torch.float64)):   # not `dev`: that is the torch.device every later cell uses
        cfg = R.Config(rule=rule, n_layer=6, d_model=128, vocab_size=512, block_size=64,
                       h=0.25 if rule != "revnet" else 1.0,
                       gamma=0.5 if rule == "blended" else 0.0)
        e = R.gradient_check(cfg, batch=2, seq=16, device=dv, dtype=dt)["worst_rel_grad_error"]
        print(f"{rule:<10}{dv:>8}{str(dt).replace('torch.',''):>10}{e:>21.2e}")


rule        device     dtype   worst rel grad err
midpoint      cuda   float32             5.77e-06
midpoint       cpu   float64             9.61e-15
blended       cuda   float32             1.76e-05
blended        cpu   float64             4.30e-14
revnet        cuda   float32             1.49e-04
revnet         cpu   float64             2.89e-13


## 3 · What width gives 20M parameters at each depth?

In [14]:
import s13_reversible as R

# Find the width that puts each depth at ~20M parameters, which is what the assignment asks
# for and what the depth sweep needs to hold fixed.
def params_at(d_model, n_layer, vocab=8192, block=512, rule="standard"):
    return R.Model(R.Config(rule=rule, d_model=d_model, n_layer=n_layer,
                            vocab_size=vocab, block_size=block)).n_params()

TARGET = 20_000_000
print(f"{'layers':>7}{'d_model':>9}{'params':>12}")
FIT = {}
for n_layer in (6, 10, 16, 24):
    best = min(range(128, 769, 64), key=lambda d: abs(params_at(d, n_layer) - TARGET))
    FIT[n_layer] = best
    print(f"{n_layer:>7}{best:>9}{params_at(best, n_layer):>12,}")
BASE_LAYERS = 10
BASE_D = FIT[BASE_LAYERS]
print(f"\nbase config for the three required runs: {BASE_LAYERS} layers, d_model {BASE_D}, "
      f"{params_at(BASE_D, BASE_LAYERS):,} params")


 layers  d_model      params
      6      448  22,197,952
     10      320  17,496,640
     16      256  17,178,880
     24      256  23,605,504

base config for the three required runs: 10 layers, d_model 320, 17,496,640 params


## 4 · Throughput and memory\n\nTwo paths, two precisions. The last column is the one that decides the plan: how long 50M tokens takes.

In [15]:
import time, torch, torch.nn.functional as F, s13_reversible as R

def bench(rule, reversible, batch, seq=512, dtype=torch.float32, steps=8,
          d_model=None, n_layer=None):
    """Tokens per second and peak memory for one configuration. Not training — timing."""
    d_model = d_model or BASE_D; n_layer = n_layer or BASE_LAYERS
    cfg = R.Config(rule=rule, d_model=d_model, n_layer=n_layer, vocab_size=8192,
                   block_size=seq, h=0.25 if rule in ("midpoint", "blended", "euler") else 1.0,
                   gamma=0.5 if rule == "blended" else 0.0)
    m = R.Model(cfg, reversible=reversible).to(device=dev, dtype=dtype)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
    x = torch.randint(8192, (batch, seq), device=dev)
    R.peak_bytes_reset(dev)
    t0 = time.time()
    for i in range(steps):
        if i == 2:                      # skip two warm-up steps, then start the clock
            if dev.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
        loss = F.cross_entropy(m(x).float().reshape(-1, 8192), x.reshape(-1))
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    dt_s = max(time.time() - t0, 1e-9)
    tps = batch * seq * max(steps - 2, 1) / dt_s
    peak = R.peak_bytes_read(dev)
    del m, opt, x
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return tps, peak, loss.item()

BATCH, SEQ = 8, 512
rows = []
print(f"{'path':<24}{'dtype':>9}{'tok/s':>11}{'peak MB':>10}{'50M tokens':>13}")
for label, rule, rev in (("standard (stored)", "standard", False),
                         ("midpoint (reversible)", "midpoint", True),
                         ("revnet (reversible)", "revnet", True)):
    for dtype in (torch.float32, torch.float16):
        try:
            tps, peak, _ = bench(rule, rev, BATCH, SEQ, dtype)
            mins = 50e6 / tps / 60
            rows.append((label, str(dtype).replace("torch.", ""), tps, peak, mins))
            print(f"{label:<24}{str(dtype).replace('torch.',''):>9}{tps:>11,.0f}"
                  f"{(peak or 0)/1e6:>10.0f}{mins:>12.1f}m")
        except Exception as exc:
            print(f"{label:<24}{str(dtype).replace('torch.',''):>9}  FAILED: {str(exc)[:50]}")


path                        dtype      tok/s   peak MB   50M tokens
standard (stored)         float32     24,973      1747        33.4m
standard (stored)         float16     85,639      1191         9.7m
midpoint (reversible)     float32     19,522       659        42.7m
midpoint (reversible)     float16     68,666       543        12.1m
revnet (reversible)       float32     44,895       536        18.6m
revnet (reversible)       float16    120,633       483         6.9m


## 5 · Maximum batch size\n\nThe assignment's third run pushes reversibility to the largest batch that fits. This is how much that is.

In [16]:
# How much batch does the saved memory actually buy? This is the assignment's third run.
def max_batch(rule, reversible, seq=512, dtype=torch.float32, cap=2048):
    b, best = 1, 0
    while b <= cap:
        try:
            bench(rule, reversible, b, seq, dtype, steps=3)
            best = b; b *= 2
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            if dev.type == "cuda":
                torch.cuda.empty_cache()
            break
    return best

DT = torch.float32
print(f"{'path':<24}{'max batch':>11}{'tokens/step':>14}")
mb = {}
for label, rule, rev in (("standard (stored)", "standard", False),
                         ("midpoint (reversible)", "midpoint", True)):
    mb[label] = max_batch(rule, rev, 512, DT)
    print(f"{label:<24}{mb[label]:>11}{mb[label]*512:>14,}")
if mb.get("standard (stored)"):
    print(f"\nreversibility buys "
          f"{mb['midpoint (reversible)']/mb['standard (stored)']:.1f}x the batch")


path                      max batch   tokens/step
standard (stored)                64        32,768
midpoint (reversible)           128        65,536

reversibility buys 2.0x the batch


## 6 · Report

In [17]:
import json
probe = {
    "gpu": torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu",
    "capability": list(torch.cuda.get_device_capability()) if dev.type == "cuda" else None,
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if dev.type == "cuda" else None,
    "bf16": torch.cuda.is_bf16_supported() if dev.type == "cuda" else False,
    "torch": torch.__version__,
    "base_layers": BASE_LAYERS, "base_d_model": BASE_D,
    "fit": FIT,
    "bench": [{"path": a, "dtype": b, "tok_s": round(c), "peak_bytes": d,
               "minutes_for_50M": round(e, 1)} for a, b, c, d, e in rows],
    "max_batch": mb,
}
print(json.dumps(probe, indent=1))
print("\n^ paste this back")


{
 "gpu": "Tesla T4",
 "capability": [
  7,
  5
 ],
 "vram_gb": 15.6,
 "bf16": true,
 "torch": "2.11.0+cu128",
 "base_layers": 10,
 "base_d_model": 320,
 "fit": {
  "6": 448,
  "10": 320,
  "16": 256,
  "24": 256
 },
 "bench": [
  {
   "path": "standard (stored)",
   "dtype": "float32",
   "tok_s": 24973,
   "peak_bytes": 1746783744,
   "minutes_for_50M": 33.4
  },
  {
   "path": "standard (stored)",
   "dtype": "float16",
   "tok_s": 85639,
   "peak_bytes": 1191391232,
   "minutes_for_50M": 9.7
  },
  {
   "path": "midpoint (reversible)",
   "dtype": "float32",
   "tok_s": 19522,
   "peak_bytes": 659115520,
   "minutes_for_50M": 42.7
  },
  {
   "path": "midpoint (reversible)",
   "dtype": "float16",
   "tok_s": 68666,
   "peak_bytes": 543109120,
   "minutes_for_50M": 12.1
  },
  {
   "path": "revnet (reversible)",
   "dtype": "float32",
   "tok_s": 44895,
   "peak_bytes": 536057344,
   "minutes_for_50M": 18.6
  },
  {
   "path": "revnet (reversible)",
   "dtype": "float16",
   "tok_s